# CosMx pathway enrichment analysis — epithelial cells

**Goal:** Identify pathways enriched in MHC II positive vs negative malignant cells
using GSEA on two gene set collections: (1) NanoString CosMx pathway annotations
and (2) MSigDB Hallmark gene sets. The Hallmark analysis is the primary figure panel;
CosMx pathway annotations provide a targeted view using panel-specific gene sets.

**Input:** `outputs/processed/epithelial.h5ad` — CosMx epithelial cells with
patient classification; `data/cosmx_pathway_annotations.csv` — NanoString CosMx
pathway gene set annotations.

**Output:** `outputs/figures/figure3-supp/figS3_hallmark_gsea.pdf`;
`outputs/tables/figure3/cosmx_gsea_hallmark.csv`;
`outputs/tables/figure3/cosmx_gsea_cosmx_pathways.csv`

## Setup

In [ ]:
from pathlib import Path
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
from scipy import stats
from statsmodels.stats.multitest import multipletests

import anndata as ad
import scanpy as sc
import decoupler as dc

# figure settings
sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

In [ ]:
# paper-wide color palettes
cmap_low_high = ['#462255FF', '#FF8811FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']
cmap_high_low = ['#FF8811FF', '#462255FF', '#9DD9D2FF', '#046E8FFF', '#D44D5CFF']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])

# inputs
epithelial_path      = repo_root / cfg['outputs']['processed'] / 'epithelial.h5ad'
pathway_annot_path   = repo_root / cfg['datasets']['cosmx']['pathway_annotations']
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])

# outputs
fig_dir   = repo_root / cfg['outputs']['figures']
table_dir = repo_root / cfg['outputs']['tables']

fig_out   = fig_dir   / 'figure3'
supp_out  = fig_dir   / 'figure3-supp'
table_out = table_dir / 'figure3'

fig_out.mkdir(parents=True, exist_ok=True)
supp_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load data

CosMx epithelial cells are loaded and subset to MHC II positive and negative
patients only. Clonal and no-malignant-cells groups are excluded from this
comparison. Gene expression is already normalized and log1p-transformed from
the upstream `mhc2_ihc_deg` notebook.

In [ ]:
# load epithelial cells
epithelial = ad.read_h5ad(epithelial_path)
print(f"Epithelial cells: {epithelial.n_obs:,} × {epithelial.n_vars} genes")

# load and merge patient classifications — not stored in epithelial.h5ad
patient_class_path = Path(cfg['datasets']['patient_classifications']['raw'])
patient_classifications = pd.read_csv(patient_class_path, index_col=0)
patient_classifications.loc['tonsil'] = ['Tonsil', 'Tonsil', 'Tonsil']
patient_classifications['patient'] = patient_classifications.index.astype(str)

epithelial.obs['patient'] = epithelial.obs['patient'].astype(str)
epithelial.obs = pd.merge(
    epithelial.obs, patient_classifications, on='patient', how='left'
)

print(epithelial.obs['patient classification'].value_counts())

# subset to MHC II pos and neg only
high_low = epithelial[
    epithelial.obs['patient classification'].isin(['class II positive', 'class II negative'])
].copy()
print(f"\nPos + neg only: {high_low.n_obs:,} cells")

## Load gene sets

Two gene set collections are used:

1. **NanoString CosMx pathway annotations** — gene sets curated specifically for
the CosMx 1000-plex panel, covering oncogenic signaling, immune, and metabolic
pathways. These provide a targeted view using only genes present in the panel.

2. **MSigDB Hallmark gene sets** — 50 well-curated, biologically coherent gene
sets representing well-defined biological states. Used as the primary figure panel
given their broad recognition in the field.

In [ ]:
# load MSigDB Hallmark gene sets via decoupler
hallmark = dc.op.hallmark(organism='human', license='academic', verbose=False)
hallmark = hallmark[~hallmark.duplicated(['source', 'target'])]
print(f"Hallmark gene sets: {hallmark['source'].nunique()}")

In [ ]:
# load NanoString CosMx pathway annotations
pathways = pd.read_csv(pathway_annot_path, index_col=0)

# reformat to long format for decoupler
long_df = (
    pathways.reset_index()
    .rename(columns={'Gene': 'genesymbol'})
    .melt(id_vars='genesymbol', var_name='geneset', value_name='Belongs')
    .query('Belongs == 1')
    .drop(columns='Belongs')
    .drop_duplicates(subset=['genesymbol', 'geneset'])
    .reset_index(drop=True)
)
long_df['collection'] = 'CosMx'
long_df['geneset'] = 'CosMx: ' + long_df['geneset'].astype(str)
print(f"CosMx pathway sets: {long_df['geneset'].nunique()}")


## Rank genes

Genes are ranked by t-test scores comparing class II positive vs negative
epithelial cells. The top 500 highly variable genes are used to focus the
analysis on the most informative features.

In [ ]:
import warnings

# compute t-test scores for class II positive vs negative
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sc.tl.rank_genes_groups(
        high_low,
        groupby='patient classification',
        method='t-test',
        key_added='t-test',
        use_raw=False,
    )

# identify highly variable genes
sc.pp.highly_variable_genes(
    high_low,
    n_top_genes=500,
    flavor='seurat',
    subset=False,
)

hvg_col = 'highly_variable' if 'highly_variable' in high_low.var.columns else 'is_highly_variable'
hvg = high_low.var.index[high_low.var[hvg_col].astype(bool)].tolist()

# build t-stats dataframe — genes ranked by score
t_stats = (
    sc.get.rank_genes_groups_df(high_low, 'class II positive', key='t-test')
    .set_index('names')
    .loc[lambda df: df.index.isin(hvg)]
    .sort_values('scores', key=np.abs, ascending=False)[['scores']]
)

print(f"Genes ranked: {len(t_stats)}")
print(t_stats.head())

## GSEA helper function

`run_gsea_collection` wraps `dc.mt.gsea` to handle gene set filtering,
minimum gene set size enforcement, and result formatting consistently
across both CosMx and Hallmark collections.

In [ ]:
def run_gsea_collection(adata, net_df, collection_name, times=200, min_genes_in_set=5):
    """
    Run decoupler GSEA for a given gene set collection.

    Parameters
    ----------
    adata : AnnData
        Normalized, log1p-transformed gene expression data.
    net_df : pd.DataFrame
        Gene sets in long format with columns ['geneset', 'genesymbol'] or ['source', 'target'].
    collection_name : str
        Label for this collection (used in progress messages).
    times : int
        Number of permutations for significance testing.
    min_genes_in_set : int
        Minimum number of genes required per gene set.

    Returns
    -------
    pd.DataFrame
        GSEA results with ES, NES, pval, padj columns.
    """
    # handle both naming conventions
    if 'geneset' in net_df.columns:
        net = net_df.rename(columns={'geneset': 'source', 'genesymbol': 'target'})
    else:
        net = net_df.copy()  # already has source/target columns

    net = net[~net.duplicated(['source', 'target'])]

    # filter to sets with enough genes
    valid_sets = (
        net.groupby('source')['target']
        .nunique()
        .loc[lambda x: x >= min_genes_in_set]
        .index
    )
    net = net[net['source'].isin(valid_sets)]
    print(f"Running GSEA for {collection_name}: {net['source'].nunique()} gene sets")

    results = dc.mt.gsea(
        adata,
        net=net,
        tmin=min_genes_in_set,
        times=times,
        seed=1,
        verbose=False,
    )
    return results

## Run GSEA

GSEA is run for both collections. Results are saved to `outputs/tables/` as
checkpoints — subsequent runs load from CSV rather than recomputing.

In [ ]:
%%time

hallmark_results_path = table_out / 'cosmx_gsea_hallmark.csv'
cosmx_results_path    = table_out / 'cosmx_gsea_cosmx_pathways.csv'

if hallmark_results_path.exists():
    print("Loading pre-computed Hallmark GSEA results...")
    gsea_hallmark = pd.read_csv(hallmark_results_path, index_col=0)
else:
    gsea_hallmark = run_gsea_collection(
        high_low, hallmark, 'hallmark', times=200, min_genes_in_set=5
    )
    gsea_hallmark.to_csv(hallmark_results_path)
    print(f"Saved → {hallmark_results_path}")

if cosmx_results_path.exists():
    print("Loading pre-computed CosMx pathway GSEA results...")
    gsea_cosmx = pd.read_csv(cosmx_results_path, index_col=0)
else:
    gsea_cosmx = run_gsea_collection(
        high_low, long_df, 'CosMx', times=200, min_genes_in_set=5
    )
    gsea_cosmx.to_csv(cosmx_results_path)
    print(f"Saved → {cosmx_results_path}")

print(gsea_hallmark)